In [ ]:
# Jupyter notebook for running inference with trained npcomposer model

from transformers import AutoTokenizer, AutoModelForCausalLM
from rdkit import Chem
from rdkit.Chem import QED
from rdkit.Contrib.SA_Score import sascorer
from rdkit.Contrib.NP_Score import npscorer
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole, rdMolDraw2D
from IPython.display import display
from PIL import Image
import io

IPythonConsole.molSize = (400, 300)
IPythonConsole.drawOptions.padding = 0.05

In [ ]:
# enter prompt for conditional generation
prompt = "<np_classifier_superclass:β-lactams> <aromatic_rings_count:1> <qed_bin:0.6<=qed<0.7> <sa_bin:6<=sa<7> "

# enter number of molecules to generate
num_molecules = 100

In [ ]:
# run inference with trained npcomposer model

tok = AutoTokenizer.from_pretrained("ralyn/NPComposer-v2", trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained("ralyn/NPComposer-v2", trust_remote_code=True).eval()

fscore = npscorer.readNPModel()

SA_scores = []
QED_scores = []
NP_scores = []

for _ in range(num_molecules): 
    x = tok(prompt, return_tensors="pt", add_special_tokens=False)
    y = model.generate(**x, max_new_tokens=200, do_sample=True, top_p=0.95, temperature=1.25)

    filtered_tok = tok.decode(y[0], skip_special_tokens=True).split(".")[0]
    print(filtered_tok)

    smiles = filtered_tok.strip()
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        # compute QED, SA, and NP scores
        qed_score = QED.qed(mol)
        sa_score = sascorer.calculateScore(mol)
        np_score = npscorer.scoreMol(mol,fscore)

        # store scores for later analysis
        QED_scores.append(float(qed_score))
        SA_scores.append(float(sa_score))
        NP_scores.append(float(np_score))

        # print scores and visualize molecule
        print(f"QED score: {qed_score:.3f}")
        print(f"SA score: {sa_score:.3f}")
        print(f"NP score: {np_score:.3f}")
        drawer = rdMolDraw2D.MolDraw2DCairo(800, 600)
        drawer.drawOptions().padding = 0.1
        drawer.DrawMolecule(mol)
        drawer.FinishDrawing()
        img = Image.open(io.BytesIO(drawer.GetDrawingText()))
        display(img)
    else:
        print("Invalid SMILES string:", smiles)

# print average scores across all generated molecules
print(f"Average SA score: {sum(SA_scores)/len(SA_scores):.3f}")
print(f"Average QED score: {sum(QED_scores)/len(QED_scores):.3f}")
print(f"Average NP score: {sum(NP_scores)/len(NP_scores):.3f}")